In [1]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import requests
import io

warnings.filterwarnings("ignore")
MSC_noon_2023=pd.read_csv("/Users/sreelakshmi.m/Downloads/NoonData_2022_28th.csv")


,id,created_at,updated_at,created_by_id,updated_by_id,uid,mail_date,gid,fleet,class,...,month_name,quarter_number,quarter_info,year,date_str,week_number,deadweight,vessel_type,vessel_name,sea_state
0,287914,NaN,NaN,NaN,NaN,446135,NaN,NaN,Fleet 2,NaN,...,January,1,2021Q1,2021,06-Jan-2021,Week 1,19874.07,Tanker,MTM North Sound,4
1,299212,NaN,NaN,NaN,NaN,470937,NaN,NaN,Fleet 4B,NaN,...,June,2,2021Q2,2021,22-Jun-2021,Week 25,39846.40,Bulker,Strategic Synergy,4
2,287915,NaN,NaN,NaN,NaN,465117,NaN,NaN,Fleet 3,NaN,...,January,1,2021Q1,2021,06-Jan-2021,Week 1,37829.00,Bulker,Strategic Fortitude,4
3,287917,NaN,NaN,NaN,NaN,437943,NaN,NaN,Fleet 3,NaN,...,January,1,2021Q1,2021,06-Jan-2021,Week 1,33089.00,Bulker,Strategic Endeavor,2
4,287918,NaN,NaN,NaN,NaN,437945,NaN,NaN,Fleet 3,NaN,...,January,1,2021Q1,2021,07-Jan-2021,Week 1,33089.00,Bulker,Strategic Endeavor,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25098,299184,NaN,NaN,NaN,NaN,469710,NaN,NaN,Fleet 4B,NaN,...,June,2,2021Q2,2021,22-Jun-2021,Week 25,39784.00,Bulker,Strategic Venture,2
25099,294462,NaN,NaN,NaN,NaN,469602,NaN,NaN,Fleet 4B,NaN,...,April,2,2021Q2,2021,14-Apr-2021,Week 15,39784.00,Bulker,Strategic Venture,4
25100,287908,NaN,NaN,NaN,NaN,455404,NaN,NaN,Fleet 1,NaN,...,January,1,2021Q1,2021,06-Jan-2021,Week 1,20856.91,Tanker,MTM Tokyo,2
25101,294467,NaN,NaN,NaN,NaN,464005,NaN,NaN,Fleet 4B,NaN,...,April,2,2021Q2,2021,14-Apr-2021,Week 15,39880.00,Bulker,Strategic Entity,2


In [2]:
MTM_noon_2022['imo'].isna().sum()

0

In [3]:
MTM_noon_2022['Date'] = pd.to_datetime(MTM_noon_2022['report_date_time']).apply(lambda x:x.strftime("%d"))
MTM_noon_2022['Month'] = pd.to_datetime(MTM_noon_2022['report_date_time']).apply(lambda x:x.strftime("%m"))
MTM_noon_2022['Year'] = pd.to_datetime(MTM_noon_2022['report_date_time']).apply(lambda x:x.strftime("%Y"))
MTM_noon_2022['Month_name'] = pd.to_datetime(MTM_noon_2022['report_date_time']).apply(lambda x:x.strftime("%B"))
MTM_noon_2022['date'] = pd.to_datetime(MTM_noon_2022["report_date_time"]).dt.date
MTM_noon_2022.dropna(subset=['gid'])
MTM_noon_2022_sort=MTM_noon_2022.sort_values(by="report_date_time")
#Total HS,LS,MGO and Total Fuel
MTM_noon_2022_sort['Total_Calculated_HS']= MTM_noon_2022_sort[['fuel_m_e_hs',"fuel_aux_hs",'fuel_boiler_hs']].sum(axis=1)
MTM_noon_2022_sort['Total_Calculated_LS'] = MTM_noon_2022_sort[["fuel_m_e_ls",'fuel_me_vlsfoh','fuel_me_vlsfol','fuel_aux_ls','fuel_aux_vlsfoh','fuel_aux_vlsfol','fuel_boiler_ls','fuel_boiler_vlsfoh','fuel_boiler_vlsfol']].fillna(0).sum(axis=1)
MTM_noon_2022_sort['Total_Calculated_MGO'] = MTM_noon_2022_sort[['fuel_m_e_mgo_hs','fuel_m_e_mgo_ls','fuel_aux_mgo_hs','fuel_aux_mgo_ls','fuel_boiler_mgo_hs','fuel_boiler_mgo_ls']].fillna(0).sum(axis=1)
MTM_noon_2022_sort['Total_Calculated_Fuel'] = MTM_noon_2022_sort[['Total_Calculated_HS' ,'Total_Calculated_LS' , 'Total_Calculated_MGO']].fillna(0).sum(axis=1)
#HS_CO2,LSCO2,MGO_CO2 and Total CO2
MTM_noon_2022_sort['HS_CO2'] = 3.114 * MTM_noon_2022_sort["Total_Calculated_HS"]
MTM_noon_2022_sort['LS_CO2'] = 3.151 * MTM_noon_2022_sort["Total_Calculated_LS"]
MTM_noon_2022_sort['MGO_CO2'] = 3.206 * MTM_noon_2022_sort["Total_Calculated_MGO"]
MTM_noon_2022_sort['Total_CO2'] = MTM_noon_2022_sort[['HS_CO2' ,'LS_CO2' ,'MGO_CO2']].fillna(0).sum(axis=1)

MTM_noon_2022_sort["fuel_m_e_calculated"]=MTM_noon_2022_sort[["fuel_m_e_hs","fuel_m_e_ls","fuel_me_vlsfoh","fuel_me_vlsfol","fuel_m_e_mgo_hs","fuel_m_e_mgo_ls"]].fillna(0).sum(axis=1)
MTM_noon_2022_sort["fuel_aux_calculated"]=MTM_noon_2022_sort[["fuel_aux_hs","fuel_aux_ls","fuel_aux_vlsfoh","fuel_aux_vlsfol","fuel_aux_mgo_hs","fuel_aux_mgo_ls"]].fillna(0).sum(axis=1)
MTM_noon_2022_sort["fuel_boiler_calculated"]=MTM_noon_2022_sort[["fuel_boiler_hs","fuel_boiler_ls","fuel_boiler_vlsfoh","fuel_boiler_vlsfol","fuel_boiler_mgo_hs","fuel_boiler_mgo_ls"]].fillna(0).sum(axis=1)
MTM_noon_2022_sort["Total_fuel_calculated"]=MTM_noon_2022_sort[["fuel_m_e_calculated","fuel_aux_calculated","fuel_boiler_calculated"]].fillna(0).sum(axis=1)


In [4]:
MTM_Noon_2021[MTM_Noon_2021['imo'].isna()]

,id,created_at,updated_at,created_by_id,updated_by_id,uid,mail_date,gid,fleet,class,...,month_name,quarter_number,quarter_info,year,date_str,week_number,deadweight,vessel_type,vessel_name,sea_state
7207,304837,NaN,NaN,NaN,NaN,366651,NaN,NaN,NaN,NaN,...,September,3,2021Q3,2021,11-Sep-2021,Week 37,NaN,NaN,NaN,5


In [5]:
MTM_Noon_2021.shape

(25103, 362)

In [6]:
Noon_2021 = MTM_Noon_2021[MTM_Noon_2021['imo'].notna()]
Noon_2021.shape

(25102, 362)

In [7]:
Noon_2021[Noon_2021['imo'].isna()]

,id,created_at,updated_at,created_by_id,updated_by_id,uid,mail_date,gid,fleet,class,...,month_name,quarter_number,quarter_info,year,date_str,week_number,deadweight,vessel_type,vessel_name,sea_state


In [8]:
Noon_2021['Date'] = pd.to_datetime(Noon_2021['report_date_time']).apply(lambda x:x.strftime("%d"))
Noon_2021['Month'] = pd.to_datetime(Noon_2021['report_date_time']).apply(lambda x:x.strftime("%m"))
Noon_2021['Year'] = pd.to_datetime(Noon_2021['report_date_time']).apply(lambda x:x.strftime("%Y"))
Noon_2021['Month_name'] = pd.to_datetime(Noon_2021['report_date_time']).apply(lambda x:x.strftime("%B"))
Noon_2021['date'] = pd.to_datetime(Noon_2021["report_date_time"]).dt.date
Noon_2021.dropna(subset=['gid'])
Noon_2021_sort=Noon_2021.sort_values(by="report_date_time")
#Total HS,LS,MGO and Total Fuel
Noon_2021_sort['Total_Calculated_HS']= Noon_2021_sort[['fuel_m_e_hs',"fuel_aux_hs",'fuel_boiler_hs']].fillna(0).sum(axis=1)
Noon_2021_sort['Total_Calculated_LS'] = Noon_2021_sort[["fuel_m_e_ls",'fuel_me_vlsfoh','fuel_me_vlsfol','fuel_aux_ls','fuel_aux_vlsfoh','fuel_aux_vlsfol','fuel_boiler_ls','fuel_boiler_vlsfoh','fuel_boiler_vlsfol']].fillna(0).sum(axis=1)
Noon_2021_sort['Total_Calculated_MGO'] = Noon_2021_sort[['fuel_m_e_mgo_hs','fuel_m_e_mgo_ls','fuel_aux_mgo_hs','fuel_aux_mgo_ls','fuel_boiler_mgo_hs','fuel_boiler_mgo_ls']].fillna(0).sum(axis=1)
Noon_2021_sort['Total_Calculated_Fuel'] = Noon_2021_sort[['Total_Calculated_HS' ,'Total_Calculated_LS' , 'Total_Calculated_MGO']].fillna(0).sum(axis=1)
#HS_CO2,LSCO2,MGO_CO2 and Total CO2
Noon_2021_sort['HS_CO2'] = 3.114 * Noon_2021_sort["Total_Calculated_HS"]
Noon_2021_sort['LS_CO2'] = 3.151 * Noon_2021_sort["Total_Calculated_LS"]
Noon_2021_sort['MGO_CO2'] = 3.206 * Noon_2021_sort["Total_Calculated_MGO"]
Noon_2021_sort['Total_CO2'] = Noon_2021_sort[['HS_CO2' ,'LS_CO2' ,'MGO_CO2']].fillna(0).sum(axis=1)

Noon_2021_sort["fuel_m_e_calculated"]=Noon_2021_sort[["fuel_m_e_hs","fuel_m_e_ls","fuel_me_vlsfoh","fuel_me_vlsfol","fuel_m_e_mgo_hs","fuel_m_e_mgo_ls"]].fillna(0).sum(axis=1)
Noon_2021_sort["fuel_aux_calculated"]=Noon_2021_sort[["fuel_aux_hs","fuel_aux_ls","fuel_aux_vlsfoh","fuel_aux_vlsfol","fuel_aux_mgo_hs","fuel_aux_mgo_ls"]].fillna(0).sum(axis=1)
Noon_2021_sort["fuel_boiler_calculated"]=Noon_2021_sort[["fuel_boiler_hs","fuel_boiler_ls","fuel_boiler_vlsfoh","fuel_boiler_vlsfol","fuel_boiler_mgo_hs","fuel_boiler_mgo_ls"]].fillna(0).sum(axis=1)
Noon_2021_sort["Total_fuel_calculated"]=Noon_2021_sort[["fuel_m_e_calculated","fuel_aux_calculated","fuel_boiler_calculated"]].fillna(0).sum(axis=1)


# Yearly Total of each Parameters

# 1. CARGO TOTAL

In [9]:
Noon_2021_sort["voyage_no"].isna().sum()

0

In [10]:
Noon_2021_sort["voyage_no"].unique()

array(['42', '54', '77', '6', '99', '8', '86', '9', '43', '49', '78.0',
       '76.0', '33', '59.0', '11', '34', '87', '52', '59', '53', '052 L',
       '55', '84', '44', '25', '104.0', '50', '88', '51', '82', '16',
       '95', '48', '33.0', '4.0', '71.0', '030L', '67V', '93', '031-B',
       '22', '20.0', '23', '85', '94', '105.0', '45', '100', '56', '7',
       '031-L', '053 B', '78.1', '26', '053 L', '12', '5.0', '46', '10',
       '77.0', '29', '76.1', '101', '35', '96', '032-B', '60.0', '106.0',
       '17', '83', '57', '72.0', '21.0', '68B', '97', '96-2', '60',
       '79.0', '36', '68L', '054 B', '34.0', '96-3', '61.0', '102', '30',
       '032-L', '96-4', '89', '80.0', '47', '96-5', '13', '033-B', '73.0',
       '45DD', '6.0', '030A', '61', '054 L', '24', '27', '030B', '96-6',
       '103', '58', '031B', '37', '033-L', '69', '22.0', '104', '81.0',
       '90', '18', '69B', '031L', '62', '62.0', '7.0', '14', '069L', '98',
       '030C', '35.0', '78', '34-B', '28', '034-B', '38'

In [11]:
Noon_2021_sort_str=Noon_2021_sort[Noon_2021_sort['voyage_no'].apply(lambda x: isinstance(x,str))]
Noon_2021_sort_str['voyage_no']=Noon_2021_sort_str['voyage_no'].str.lstrip('0')
Noon_2021_sort_str['voyage_no']=Noon_2021_sort_str['voyage_no'].str.rstrip('.0')

Noon_2021_sort_float=Noon_2021_sort[Noon_2021_sort['voyage_no'].apply(lambda x: isinstance(x,float))]
Noon_2021_sort_float['voyage_no']=Noon_2021_sort_float['voyage_no'].astype(int).astype(str)

Noon_2021_sort_int=Noon_2021_sort[Noon_2021_sort['voyage_no'].apply(lambda x: isinstance(x,int))]
Noon_2021_sort_int['voyage_no']=Noon_2021_sort_int['voyage_no'].astype(int).astype(str)

Noon_2021_sort=pd.concat([Noon_2021_sort_str,Noon_2021_sort_float,Noon_2021_sort_int])
Noon_2021_sort=Noon_2021_sort.sort_values(by=["report_date_time"])

In [12]:
Noon_2021_sort["voyage_no"].unique()

array(['42', '54', '77', '6', '99', '8', '86', '9', '43', '49', '48', '5',
       '88', '51', '82', '16', '33', '22', '4', '71', '30L', '67V', '93',
       '31-B', '2', '104', '95', '44', '78', '76', '59', '11', '34', '87',
       '52', '84', '55', '52 L', '25', '53', '23', '94', '85', '105',
       '45', '1', '7', '56', '31-L', '53 B', '78.1', '26', '53 L', '12',
       '46', '76.1', '29', '101', '35', '96', '32-B', '106', '17', '83',
       '57', '72', '21', '68B', '97', '96-2', '79', '36', '68L', '54 B',
       '96-3', '61', '102', '3', '32-L', '96-4', '89', '47', '96-5', '13',
       '33-B', '73', '45DD', '30A', '54 L', '24', '27', '30B', '96-6',
       '103', '58', '31B', '37', '33-L', '69', '81', '18', '69B', '31L',
       '62', '14', '69L', '98', '30C', '34-B', '28', '38', '91', '32B',
       '63', '74', '19', '15', '32L', '56L', '34-L', '92', '70B', '39',
       '', '35-B', '70L', '33B', '71B', '53B', '64', '71L', '35L', '63B',
       '31', '36B', '57 B', '47-4', '33L', '107', 

In [13]:
MTM_noon_2022_sort["voyage_no"].isna().sum()

10

In [14]:
MTM_noon_2022_sort['voyage_no'] = MTM_noon_2022_sort.groupby(['vessel_name'])['voyage_no'].transform(lambda v: v.ffill())

In [15]:
MTM_noon_2022_str=MTM_noon_2022_sort[MTM_noon_2022_sort['voyage_no'].apply(lambda x: isinstance(x,str))]
MTM_noon_2022_str['voyage_no']=MTM_noon_2022_str['voyage_no'].str.lstrip('0')
MTM_noon_2022_str['voyage_no']=MTM_noon_2022_str['voyage_no'].str.rstrip('.0')
MTM_noon_2022_float=MTM_noon_2022_sort[MTM_noon_2022_sort['voyage_no'].apply(lambda x: isinstance(x,float))]
MTM_noon_2022_float['voyage_no']=MTM_noon_2022_float['voyage_no'].astype(int).astype(str)
MTM_noon_2022_int=MTM_noon_2022_sort[MTM_noon_2022_sort['voyage_no'].apply(lambda x: isinstance(x,int))]
MTM_noon_2022_int['voyage_no']=MTM_noon_2022_int['voyage_no'].astype(int).astype(str)

MTM_noon_2022_sort_new=pd.concat([MTM_noon_2022_str,MTM_noon_2022_float,MTM_noon_2022_int])
MTM_noon_2022_sort_new=MTM_noon_2022_sort_new.sort_values(by=["report_date_time"])

In [16]:
MTM_noon_2022_sort_new["voyage_no"].isna().sum()

0

In [17]:
MTM_noon_2022_sort_new["cargo_total"].isna().sum()

0

In [18]:
Noon_2021_sort["cargo_total"].isna().sum()

0

In [19]:
#for getting mean of each voyage 
cargo_vessel_mean_2021=Noon_2021_sort.groupby(["imo","voyage_no"])[["cargo_total"]].mean().reset_index()
Total_cargo_2021=cargo_vessel_mean_2021[["cargo_total"]].sum().reset_index()
Total_cargo_2021=Total_cargo_2021[0]/(10**6)
Total_cargo_2021.reset_index()

cargo_vessel_mean_2022=MTM_noon_2022_sort_new.groupby(["imo","voyage_no"])[["cargo_total"]].mean().reset_index()
Total_cargo_2022=cargo_vessel_mean_2022[["cargo_total"]].sum().reset_index()
Total_cargo_2022=Total_cargo_2022[0]/(10**6)
Total_cargo_2022.reset_index()

,index,0
0,0,7.789923


# 2.  MILES BY GPS

In [20]:
MTM_noon_2022_sort_new.columns.values

array(['id', 'created_at', 'updated_at', 'created_by_id', 'updated_by_id',
       'uid', 'mail_date', 'gid', 'fleet', 'class', 'vessel_id', 'imo',
       'voyage_name', 'voy', 'voy_condition', 'voyage_no', 'voyage_code',
       'stoppages', 'stoppages_hh', 'stoppages_mm', 'report_type',
       'status', 'lattitude', 'longitude', 'corrected_date',
       'report_month', 'report_date_time', 'report_date_time_offset',
       'time_zone', 'port_code', 'port', 'depart_arrival',
       'departure_date', 'dep_port', 'arrival_date', 'arrival_port',
       'arrival_port_agent', 'date_berthed', 'cosp_date', 'sbe_date',
       'fwe_date', 'cosp_time', 'eosp_date', 'date_anchored',
       'charter_party_cons', 'charter_party_speed', 'charterer_name',
       'cargo_total', 'cargo_total_teu', 'teu_full', 'teu_empty',
       'laden_20_ft', 'laden_40_ft', 'laden_45_ft', 'empty_20_ft',
       'empty_40_ft', 'empty_45_ft', 'refr_laden_20_ft',
       'refr_laden_40_ft', 'total_cargo_loaded', 'total_cargo

In [21]:
Noon_2021_sort["miles_by_gps"].isna().sum()

491

In [22]:
Noon_2021_sort["miles_by_gps"] = Noon_2021_sort["miles_by_gps"].replace(np.nan, 0)

In [23]:
Noon_2021_sort["miles_by_gps"].isna().sum()

0

In [24]:
MTM_noon_2022_sort_new["miles_by_gps"].isna().sum()

0

In [25]:
#for getting sum of distance in million nautical miles 


distance_2021=Noon_2021_sort.groupby(["imo"])[["miles_by_gps"]].sum().reset_index()
Total_distance_2021=distance_2021[["miles_by_gps"]].sum().reset_index()
Total_distance_2021=Total_distance_2021[0]/(10**6)
Total_distance_2021.reset_index()

distance_2022=MTM_noon_2022_sort_new.groupby(["imo"])[["miles_by_gps"]].sum().reset_index()
Total_distance_2022=distance_2022[["miles_by_gps"]].sum().reset_index()
Total_distance_2022=Total_distance_2022[0]/(10**6)
Total_distance_2022.reset_index()

,index,0
0,0,3.298408


# 3.  Voyages Laden and Ballast

In [26]:
Noon_2021_sort["voy_condition"].isna().sum()

0

In [27]:
MTM_noon_2022_sort_new["voy_condition"].isna().sum()

0

In [28]:
Total_voy_time_2021=Noon_2021_sort[["vessel_name","report_date_time","voy_condition","voyage_no"]]
Total_voy_time_2021['report_date_time'] = pd.to_datetime(Total_voy_time_2021['report_date_time'], errors='coerce')
Total_voy_time_2021=Total_voy_time_2021.sort_values(by=["vessel_name","report_date_time"])
Total_voy_time_2021["Time"]=Total_voy_time_2021.groupby(["vessel_name"])[["report_date_time"]].diff()
Total_voy_time_2021['Time']  = Total_voy_time_2021.Time.shift(-1)
Total_voy_time_2021['Time_diff_hour'] = Total_voy_time_2021["Time"]/ pd.Timedelta(hours=1)
Total_voy_time_2021['Time_diff_hour'] .mask(Total_voy_time_2021['Time_diff_hour']  < 0 ,0, inplace=True)
Total_loaded_voy_time_2021=Total_voy_time_2021[Total_voy_time_2021["voy_condition"]=="Laden"]
Total_loaded_voy_time_2021=Total_loaded_voy_time_2021.groupby("voy_condition")["Time_diff_hour"].sum().reset_index()
Total_loaded_voy_time_2021["Total_days"]=(Total_loaded_voy_time_2021["Time_diff_hour"]/(24))
Total_loaded_voy_time_2021

,voy_condition,Time_diff_hour,Total_days
0,Laden,314681.75,13111.739583


In [29]:
Total_ballast_voy_time_2021=Total_voy_time_2021[Total_voy_time_2021["voy_condition"]=="Ballast"]
Total_ballast_voy_time_2021=Total_ballast_voy_time_2021.groupby("voy_condition")["Time_diff_hour"].sum().reset_index()
Total_ballast_voy_time_2021["Total_days"]=(Total_ballast_voy_time_2021["Time_diff_hour"]/(24))
Total_ballast_voy_time_2021

,voy_condition,Time_diff_hour,Total_days
0,Ballast,153842.833333,6410.118056


In [30]:
Total_voy_time_2022=MTM_noon_2022_sort_new[["vessel_name","report_date_time","voy_condition","voyage_no"]]
Total_voy_time_2022['report_date_time'] = pd.to_datetime(Total_voy_time_2022['report_date_time'], errors='coerce')
Total_voy_time_2022=Total_voy_time_2022.sort_values(by=["vessel_name","report_date_time"])
Total_voy_time_2022["Time"]=Total_voy_time_2022.groupby(["vessel_name"])[["report_date_time"]].diff()
Total_voy_time_2022['Time']  = Total_voy_time_2022.Time.shift(-1)
Total_voy_time_2022['Time_diff_hour'] = Total_voy_time_2022["Time"]/ pd.Timedelta(hours=1)
Total_voy_time_2022['Time_diff_hour'] .mask(Total_voy_time_2022['Time_diff_hour']  < 0 ,0, inplace=True)
Total_loaded_voy_time_2022=Total_voy_time_2022[Total_voy_time_2022["voy_condition"]=="Laden"]
Total_loaded_voy_time_2022=Total_loaded_voy_time_2022.groupby("voy_condition")["Time_diff_hour"].sum().reset_index()
Total_loaded_voy_time_2022["Total_days"]=(Total_loaded_voy_time_2022["Time_diff_hour"]/(24))
Total_loaded_voy_time_2022

,voy_condition,Time_diff_hour,Total_days
0,Laden,335118.95,13963.289583


In [31]:
Total_ballast_voy_time_2022=Total_voy_time_2022[Total_voy_time_2022["voy_condition"]=="Ballast"]
Total_ballast_voy_time_2022=Total_ballast_voy_time_2022.groupby("voy_condition")["Time_diff_hour"].sum().reset_index()
Total_ballast_voy_time_2022["Total_days"]=(Total_ballast_voy_time_2022["Time_diff_hour"]/(24))
Total_ballast_voy_time_2022

,voy_condition,Time_diff_hour,Total_days
0,Ballast,134551.283333,5606.303472


# 4.  Changing unit of Fuel

In [36]:
#for getting sum of fuel in kilo tonnes 
#fuel_hs_calculated

#fuel_ls_calculated

fuel_ls_2021=Noon_2021_sort.groupby(["imo"])[["Total_Calculated_LS"]].sum().reset_index()
Total_fuel_ls_2021=fuel_ls_2021[["Total_Calculated_LS"]].sum().reset_index()
Total_fuel_ls_2021=Total_fuel_ls_2021[0]/(10**3)
Total_fuel_ls_2021.reset_index()

fuel_ls_2022=MTM_noon_2022_sort.groupby(["imo"])[["Total_Calculated_LS"]].sum().reset_index()
Total_fuel_ls_2022=fuel_ls_2022[["Total_Calculated_LS"]].sum().reset_index()
Total_fuel_ls_2022=Total_fuel_ls_2022[0]/(10**3)
Total_fuel_ls_2022.reset_index()

#fuel_mgo_calculated

fuel_mgo_2021=Noon_2021_sort.groupby(["imo"])[["Total_Calculated_MGO"]].sum().reset_index()
Total_fuel_mgo_2021=fuel_mgo_2021[["Total_Calculated_MGO"]].sum().reset_index()
Total_fuel_mgo_2021=Total_fuel_mgo_2021[0]/(10**3)
Total_fuel_mgo_2021.reset_index()

fuel_mgo_2022=MTM_noon_2022_sort.groupby(["imo"])[["Total_Calculated_MGO"]].sum().reset_index()
Total_fuel_mgo_2022=fuel_mgo_2022[["Total_Calculated_MGO"]].sum().reset_index()
Total_fuel_mgo_2022=Total_fuel_mgo_2022[0]/(10**3)
Total_fuel_mgo_2022.reset_index()

#fuel_me_calculated

fuel_me_2021=Noon_2021_sort.groupby(["imo"])[["fuel_m_e_calculated"]].sum().reset_index()
Total_fuel_me_2021=fuel_me_2021[["fuel_m_e_calculated"]].sum().reset_index()
Total_fuel_me_2021=Total_fuel_me_2021[0]/(10**3)
Total_fuel_me_2021.reset_index()

fuel_me_2023=MSC_noon_2023_sort.groupby(["imo"])[["fuel_m_e_calculated"]].sum().reset_index()
Total_fuel_me_2023=fuel_me_2023[["fuel_m_e_calculated"]].sum().reset_index()
Total_fuel_me_2023=Total_fuel_me_2023[0]/(10**3)
Total_fuel_me_2023.reset_index()

#fuel_aux_calculated

fuel_aux_2023=MSC_noon_2023_sort.groupby(["imo"])[["fuel_aux_calculated"]].sum().reset_index()
Total_fuel_aux_2023=fuel_aux_2023[["fuel_aux_calculated"]].sum().reset_index()
Total_fuel_aux_2023=Total_fuel_aux_2023[0]/(10**3)
Total_fuel_aux_2023.reset_index()

#fuel_boiler_calculated

fuel_boiler_2023=MSC_noon_2023_sort.groupby(["imo"])[["fuel_boiler_calculated"]].sum().reset_index()
Total_fuel_boiler_202=fuel_boiler_2022[["fuel_boiler_calculated"]].sum().reset_index()
Total_fuel_boiler_2022=Total_fuel_boiler_2022[0]/(10**3)
Total_fuel_boiler_2022.reset_index()

#fuel_total_calculated

fuel_2022=MSC_noon_2023_sort.groupby(["imo"])[["Total_fuel_calculated"]].sum().reset_index()
Total_fuel_2022=fuel_2022[["Total_fuel_calculated"]].sum().reset_index()
Total_fuel_2022=Total_fuel_2022[0]/(10**3)
Total_fuel_2022.reset_index()

#fuel_CO2_calculated

fuel_CO2_2021=MSC_noon_2023_sort.groupby(["imo"])[["Total_CO2"]].sum().reset_index()
Total_CO2_2021=fuel_CO2_2021[["Total_CO2"]].sum().reset_index()
Total_CO2_2021=Total_CO2_2021[0]/(10**6)
Total_CO2_2021.reset_index()

fuel_CO2_2022=MSC_noon_2023_sort.groupby(["imo"])[["Total_CO2"]].sum().reset_index()
Total_CO2_2022=fuel_CO2_2022[["Total_CO2"]].sum().reset_index()
Total_CO2_2022=Total_CO2_2022[0]/(10**6)
Total_CO2_2022.reset_index()

#Fresh Water Produced

FW_Pro_2022=MSC_noon_2023_sort.groupby(["imo"])[["generated_fresh_water"]].sum().reset_index()
Total_FW_Pro_2022=FW_Pro_2022[["generated_fresh_water"]].sum().reset_index()
Total_FW_Pro_2022=Total_FW_Pro_2022[0]/(10**3)
Total_FW_Pro_2022.reset_index()

#Fresh Water Consumed


FW_Con_2022=MSC_noon_2023_sort.groupby(["imo"])[["consumed_fresh_water"]].sum().reset_index()
Total_FW_Con_2022=FW_Con_2022[["consumed_fresh_water"]].sum().reset_index()
Total_FW_Con_2022=Total_FW_Con_2022[0]/(10**3)
Total_FW_Con_2022.reset_index()

#Steaming Time

Steam_2021=Noon_2021_sort.groupby(["imo"])[["m_e_fuel_only_steaming_time"]].sum().reset_index()
Total_Steam_2021=Steam_2021[["m_e_fuel_only_steaming_time"]].sum().reset_index()
Total_Steam_2021=Total_Steam_2021[0]/(24)
Total_Steam_2021.reset_index()

Steam_2022=MTM_noon_2022_sort.groupby(["imo"])[["m_e_fuel_only_steaming_time"]].sum().reset_index()
Total_Steam_2022=Steam_2022[["m_e_fuel_only_steaming_time"]].sum().reset_index()
Total_Steam_2022=Total_Steam_2022[0]/(24)
Total_Steam_2022.reset_index()



,index,0
0,0,11870.340965


In [37]:

Total_fuel_LS= {'Parameter':['Total LS Fuel'], '2021':Total_fuel_ls_2021[0].sum(), '2022':Total_fuel_ls_2022[0].sum()}
Total_fuel_LS= pd.DataFrame(Total_fuel_LS)

Total_fuel_MGO= {'Parameter':['Total MGO Fuel'], '2021':Total_fuel_mgo_2021[0].sum(), '2022':Total_fuel_mgo_2022[0].sum()}
Total_fuel_MGO= pd.DataFrame(Total_fuel_MGO)

Total_fuel= {'Parameter':['Total Fuel'], '2021':Total_fuel_2021[0].sum(), '2022':Total_fuel_2022[0].sum()}
Total_fuel= pd.DataFrame(Total_fuel)

Total_ME_fuel= {'Parameter':['Total ME Fuel'], '2021':Total_fuel_me_2021[0].sum(), '2022':Total_fuel_me_2022[0].sum()}
Total_ME_fuel= pd.DataFrame(Total_ME_fuel)

Total_AUX_fuel= {'Parameter':['Total AUX Fuel'], '2021':Total_fuel_aux_2021[0].sum(), '2022':Total_fuel_aux_2022[0].sum()}
Total_AUX_fuel= pd.DataFrame(Total_AUX_fuel)

Total_BOI_fuel= {'Parameter':['Total BOILER Fuel'], '2021':Total_fuel_boiler_2021[0].sum(), '2022':Total_fuel_boiler_2022[0].sum()}
Total_BOI_fuel= pd.DataFrame(Total_BOI_fuel)

Total_CO2= {'Parameter':['Total CO2'], '2021':Total_CO2_2021[0].sum(), '2022':Total_CO2_2022[0].sum()}
Total_CO2= pd.DataFrame(Total_CO2)

#distance is calculating using above cell procedure(sum of vessel,total sum,divide it with 10^6 - Million nautical miles)
Total_Distance= {'Parameter':['Total Distance'], '2021':Total_distance_2021[0], '2022':Total_distance_2022[0].sum()}
Total_Distance= pd.DataFrame(Total_Distance)

Total_fresh_pro= {'Parameter':['Fresh Water Produced'], '2021':Total_FW_Pro_2021[0].sum(), '2022':Total_FW_Pro_2022[0].sum()}
Total_fresh_pro= pd.DataFrame(Total_fresh_pro)

Total_fresh_con= {'Parameter':['Fresh Water Consumed'], '2021':Total_FW_Con_2021[0].sum(), '2022':Total_FW_Con_2022[0].sum()}
Total_fresh_con= pd.DataFrame(Total_fresh_con)

Total_voy= {'Parameter':['Total Voyages'], '2021':Noon_2021_sort["voyage_no"].nunique(), '2022':MTM_noon_2022_sort_new["voyage_no"].nunique()}
Total_voy= pd.DataFrame(Total_voy)

Total_vsl= {'Parameter':['Number of Vessel'], '2021':Noon_2021_sort["imo"].nunique(), '2022':MTM_noon_2022_sort_new["imo"].nunique()}
Total_vsl= pd.DataFrame(Total_vsl)
#Total_vsl['2020'] = Total_vsl['2020'].fillna(0).astype(int)

Total_steaming_time= {'Parameter':['Total Steaming Days'], '2021':Total_Steam_2021[0].sum(), '2022':Total_Steam_2022[0].sum()}
Total_steaming_time= pd.DataFrame(Total_steaming_time)

#cargo is calculating using above cell procedure(mean of vessel that is in voyage wise,total sum,multiple it with 10^6 - Million tonnes)
Total_Cargo= {'Parameter':['Total Cargo'], '2021':Total_cargo_2021[0], '2022':Total_cargo_2022[0]}
Total_Cargo= pd.DataFrame(Total_Cargo)

Total_ballast_voy_time= {'Parameter':['Total Ballast Days'], '2021':Total_ballast_voy_time_2021["Total_days"], '2022':Total_ballast_voy_time_2022["Total_days"]}
Total_ballast_voy_time= pd.DataFrame(Total_ballast_voy_time)

Total_Laden_voy_time= {'Parameter':['Total Loaded Days'], '2021':Total_loaded_voy_time_2021["Total_days"], '2022':Total_loaded_voy_time_2022["Total_days"]}
Total_Laden_voy_time= pd.DataFrame(Total_Laden_voy_time)

Avg_ballast_voy_time= {'Parameter':['Average Ballast Days'], '2021':Total_ballast_voy_time_2021["Avg_ballast_days"], '2022':Total_ballast_voy_time_2022["Avg_ballast_days"]}
Avg_ballast_voy_time= pd.DataFrame(Avg_ballast_voy_time)

Avg_Laden_voy_time= {'Parameter':['Average Loaded Days'], '2021':Total_loaded_voy_time_2021["Avg_load_days"], '2022':Total_loaded_voy_time_2022["Avg_load_days"]}
Avg_Laden_voy_time= pd.DataFrame(Avg_Laden_voy_time)

Total_Parameters=pd.concat([Total_vsl,Total_fuel_LS,Total_fuel_MGO,Total_fuel,Total_ME_fuel,Total_AUX_fuel,Total_BOI_fuel,Total_CO2,Total_Cargo,Total_Distance,Total_fresh_pro,Total_fresh_con,Total_voy,Total_steaming_time,Total_ballast_voy_time,Total_Laden_voy_time,Avg_ballast_voy_time,Avg_Laden_voy_time])

Total_Parameters=Total_Parameters.reset_index()
Total_Parameters=Total_Parameters.drop(['index'], axis=1)

In [38]:
unit_parameters = [" ","Kilotonne","Kilotonne","Kilotonne","Kilotonne","Kilotonne","Kilotonne","Million Tonne","Million Tonne","Million Nautical Miles","Kilotonne","Kilotonne","","Day","Day","Day","Day","Day"]
parameter_unit = pd.DataFrame(unit_parameters, columns=['Unit'])
Total_Parameter_1=pd.concat([Total_Parameters,parameter_unit],axis=1)
Total_Parameter_1

,Parameter,2021,2022,Unit
0,Number of Vessel,56.000000,56.000000,
1,Total LS Fuel,217.089890,215.701560,Kilotonne
2,Total MGO Fuel,36.257710,37.541200,Kilotonne
3,Total Fuel,253.347600,253.242760,Kilotonne
4,Total ME Fuel,190.643690,190.168660,Kilotonne
5,Total AUX Fuel,39.994470,39.998590,Kilotonne
6,Total BOILER Fuel,22.709440,23.075510,Kilotonne
7,Total CO2,0.800292,0.800033,Million Tonne
8,Total Cargo,7.383651,7.789923,Million Tonne
9,Total Distance,3.244236,3.298408,Million Nautical Miles


In [39]:
Total_Parameter_1["%change(2021-2022)"]=(Total_Parameter_1["2022"]-Total_Parameter_1["2021"])*(100)/(Total_Parameter_1["2021"])
#Total_Parameter_1["Projected 2022"]=(Total_Parameter_1["2022"]*12)/(7)
#Total_Parameter_1["%change projected22"]=(Total_Parameter_1["Projected 2022"]-Total_Parameter_1["2021"])*(100)/(Total_Parameter_1["2021"])
Total_Parameter_1=Total_Parameter_1[['Parameter', "Unit", '2021',"2022",'%change(2021-2022)' ]]
Total_Parameter_1=Total_Parameter_1.round(decimals = 2)
Total_Parameter_1

,Parameter,Unit,2021,2022,%change(2021-2022)
0,Number of Vessel,,56.00,56.00,0.00
1,Total LS Fuel,Kilotonne,217.09,215.70,-0.64
2,Total MGO Fuel,Kilotonne,36.26,37.54,3.54
3,Total Fuel,Kilotonne,253.35,253.24,-0.04
4,Total ME Fuel,Kilotonne,190.64,190.17,-0.25
5,Total AUX Fuel,Kilotonne,39.99,40.00,0.01
6,Total BOILER Fuel,Kilotonne,22.71,23.08,1.61
7,Total CO2,Million Tonne,0.80,0.80,-0.03
8,Total Cargo,Million Tonne,7.38,7.79,5.50
9,Total Distance,Million Nautical Miles,3.24,3.30,1.67


In [40]:
dict_Total_Parameter = Total_Parameter_1.to_json(orient = 'records')
dict_Total_Parameter

'[{"Parameter":"Number of Vessel","Unit":" ","2021":56.0,"2022":56.0,"%change(2021-2022)":0.0},{"Parameter":"Total LS Fuel","Unit":"Kilotonne","2021":217.09,"2022":215.7,"%change(2021-2022)":-0.64},{"Parameter":"Total MGO Fuel","Unit":"Kilotonne","2021":36.26,"2022":37.54,"%change(2021-2022)":3.54},{"Parameter":"Total Fuel","Unit":"Kilotonne","2021":253.35,"2022":253.24,"%change(2021-2022)":-0.04},{"Parameter":"Total ME Fuel","Unit":"Kilotonne","2021":190.64,"2022":190.17,"%change(2021-2022)":-0.25},{"Parameter":"Total AUX Fuel","Unit":"Kilotonne","2021":39.99,"2022":40.0,"%change(2021-2022)":0.01},{"Parameter":"Total BOILER Fuel","Unit":"Kilotonne","2021":22.71,"2022":23.08,"%change(2021-2022)":1.61},{"Parameter":"Total CO2","Unit":"Million Tonne","2021":0.8,"2022":0.8,"%change(2021-2022)":-0.03},{"Parameter":"Total Cargo","Unit":"Million Tonne","2021":7.38,"2022":7.79,"%change(2021-2022)":5.5},{"Parameter":"Total Distance","Unit":"Million Nautical Miles","2021":3.24,"2022":3.3,"%chan